In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv("data/loan_approval.csv")

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
print(df.duplicated().sum())

In [ ]:
df.tail()

In [ ]:
df['city'].unique()

In [ ]:
df['city'].nunique()

In [ ]:
df.isnull().sum()

In [ ]:
bool=df.select_dtypes(include='boolean')
bool


In [ ]:
loan_approved_dict={True:1,False:0}
bool['loan_approved']=bool['loan_approved'].map(loan_approved_dict)

In [ ]:
bool

In [ ]:
sns.countplot(x='loan_approved',data=df)
plt.show()

In [ ]:
sns.histplot([df['income']])

In [ ]:
sns.histplot([df['credit_score']])

In [ ]:
sns.kdeplot([df['income']])

In [ ]:
num_cols=df.select_dtypes(include=np.number).columns

for col in num_cols:
    sns.boxplot(x=df[col])
    plt.title(col)
    plt.show()

In [ ]:
sns.boxplot(x="loan_approved", y="credit_score", data=df)
plt.show()

In [ ]:
sns.scatterplot(
    x="loan_amount",
    y="years_employed",
    hue="loan_approved",
    data=df
)
plt.show()

In [ ]:
sns.scatterplot(
    x="income",
    y="loan_amount",
    hue="loan_approved",
    data=df
)
plt.show()

In [ ]:
df.drop(['name','city'],axis=1,inplace=True)

In [ ]:
x=df[['income',
      'credit_score',
      'loan_amount',
      'years_employed',
      ]]

y=df['loan_approved']        # 'points' was excluded to evaluate the predictive power
# of the remaining financial features and make the model
# rely on directly interpretable applicant information.

In [ ]:
#splitting datset
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
#SVM model
from sklearn.svm import SVC,SVR
sv=SVC()
sv.fit(x_train,y_train)
print("test score:",sv.score(x_test,y_test))
print("train score:",sv.score(x_train,y_train))

In [ ]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression
log=LogisticRegression()      
log.fit(x_train,y_train)
print("test score",log.score(x_test,y_test))
print("train score",log.score(x_train,y_train))

In [ ]:
# Decision tree classifier
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
dt=DecisionTreeClassifier()
dt.fit(x_train,y_train)
print("test score",dt.score(x_test,y_test))
print("train score",dt.score(x_train,y_train))

In [ ]:
dt.get_params()

In [ ]:
#Tuned Decision Tree Classifier
dt_tuned = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
dt_tuned.fit(x_train,y_train)
print("test score",dt_tuned.score(x_test,y_test))
print("train score",dt_tuned.score(x_train,y_train))

In [ ]:
#Random Forest
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_estimators=100
)
rf.fit(x_train,y_train)
print("test score",rf.score(x_test,y_test))
print("train score",rf.score(x_train,y_train))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,roc_auc_score

# Function to calculate and print evaluation metrics
def evaluate_model(model, x_test, y_test, model_name):   
    y_pred = model.predict(x_test)
    print(f"\n=== {model_name} ===")
    
     # Classification report
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    # Confusion matrix
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    # Calculate and print accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy:.4f}")
    
    
    try:
        roc_auc = roc_auc_score(y_test, y_pred)
        print(f"ROC AUC Score: {roc_auc:.4f}")
    except (ValueError, AttributeError):
        print("ROC AUC Score: Not applicable")
# Evaluate Logistic Regression
evaluate_model(log, x_test, y_test, "Logistic Regression")
# Evaluate  Tuned Decision Tree Classifier
evaluate_model(dt_tuned, x_test, y_test, "Decision Tree Tuned")
# Evaluate Support Vector Classifier 
evaluate_model(sv, x_test, y_test, "Support Vector Classifier")
# Evaluate Random Forest Classifier 
evaluate_model(rf, x_test, y_test, "Random Forest Classifier")
 

In [ ]:
# Get predictions from each model on x_test
y_preds = [
    log.predict(x_test),
    dt_tuned.predict(x_test),
    sv.predict(x_test),
    rf.predict(x_test)
]

models = ['Logistic Regression', 'Decision Tree Tuned', 'Support Vector classifier','Random Forest Classifier']

# Calculate accuracies for each model
accuracies = [accuracy_score(y_test, y_pred) for y_pred in y_preds]

# Plot accuracy comparison
plt.figure(figsize=(12, 6))
sns.barplot(x=models, y=accuracies)
plt.title('Model Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.show()


In [ ]:
best_model_index = np.argmax(accuracies)
best_model_name = models[best_model_index]

print(f"The best model is: {best_model_name} with an accuracy of {accuracies[best_model_index]:.4f}")

In [ ]:
# Get the best model object
model_objects = [
    log,
    dt_tuned,
    sv,
    rf
]
best_model = model_objects[best_model_index]

# Predict using the best model
y_test_pred = best_model.predict(x_test)

# Print predictions vs actual labels
print("\nPredictions vs Actual Labels:")
for i in range(len(y_test)):
    print(f"Sample {i+1}: Prediction = {y_test_pred[i]}, Actual = {y_test.iloc[i]}")

# Calculate and display accuracy
accuracy = accuracy_score(y_test, y_test_pred)
print(f"\nAccuracy: {accuracy * 100:.2f}%")



In [ ]:
import joblib
joblib.dump(best_model, "model/best_model.lb")
model = joblib.load("model/best_model.lb")
print(model)